# GE-Bot-1: Real PlantVillage MobileNetV2 Leaf-Disease Training Pipeline
**Project:** GE-Bot-1 Autonomous Organic Farming Robot

This Google Colab notebook trains a **MobileNetV2 transfer-learning model** on the **PlantVillage dataset** (54,000+ images, 38 classes across 14 crops) and exports the trained model directly to **TensorFlow.js format** for edge deployment in the browser.

In [ ]:
# Step 1: Install dependencies and TensorFlow.js converter
!pip install -q tensorflowjs kagglehub

In [ ]:
# Step 2: Download the PlantVillage dataset
import kagglehub
import os

print("Downloading PlantVillage dataset from Kaggle...")
path = kagglehub.dataset_download("emmarex/plantdisease")
print("Path to dataset files:", path)

# Locate the PlantVillage folder inside the download path
dataset_dir = None
for root, dirs, files in os.walk(path):
    if "PlantVillage" in dirs or "color" in dirs:
        dataset_dir = os.path.join(root, "PlantVillage" if "PlantVillage" in dirs else "color")
        break

if not dataset_dir:
    dataset_dir = path
print(f"Selected dataset directory: {dataset_dir}")

In [ ]:
# Step 3: Configure Data Generators with Heavy Outdoor Augmentation
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Outdoor field robust augmentations (illumination jitter, vantage shift, zoom)
train_datagen = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=35,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.25,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.75, 1.25],
    fill_mode='reflect'
)

val_datagen = ImageDataGenerator(
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"Found {train_gen.num_classes} classes and {train_gen.samples} training samples.")
class_indices = train_gen.class_indices
with open("classes.json", "w") as f:
    json.dump(class_indices, f, indent=2)
print("Classes mapping saved to classes.json")

In [ ]:
# Step 4: Build MobileNetV2 Transfer Learning Architecture
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = layers.Rescaling(scale=1./127.5, offset=-1.0)(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(train_gen.num_classes, activation='softmax', name="disease_predictions")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="GE_Bot_PlantVillage_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_acc')]
)
model.summary()

In [ ]:
# Step 5: Train Phase 1 (Frozen Base)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=8,
    callbacks=callbacks
)

In [ ]:
# Step 6: Train Phase 2 (Fine-tuning Top 30 MobileNetV2 Layers)
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_acc')]
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=6,
    callbacks=callbacks
)

In [ ]:
# Step 7: Export Model to TensorFlow.js Web Format
import subprocess

model.save("plantvillage_mobilenetv2_final.keras")
print("Keras model saved successfully.")

!tensorflowjs_converter --input_format=keras --output_format=tfjs_layers_model --quantization_bytes=2 plantvillage_mobilenetv2_final.keras ./tfjs_model

print("Exported TF.js Model Files:")
!ls -lh ./tfjs_model

In [ ]:
# Step 8: Package and Download for GE-Bot-1 Frontend
from google.colab import files
import shutil

shutil.make_archive("ge_bot_plant_disease_model", 'zip', "./tfjs_model")
print("Model archive created: ge_bot_plant_disease_model.zip")
files.download("ge_bot_plant_disease_model.zip")
files.download("classes.json")